In [8]:
#Test connection to Google Books API 
import requests
from getpass import getpass

# Securely prompt for API key (input is hidden)
api_key = getpass("Enter your Google Books API key: ")

url = "https://www.googleapis.com/books/v1/volumes"
params = {
    "q": "fiction",
    "maxResults": 40,
    "key": api_key  # Add API key to request
}

response = requests.get(url, params=params)
print(f"Status Code: {response.status_code}")
data = response.json()

# Check if request was successful
if response.status_code == 200 and 'items' in data:
    print(f"Number of items: {len(data['items'])}")
else:
    print(f"Error: {data.get('error', {}).get('message', 'Unknown error')}")

Status Code: 200
Number of items: 20


In [9]:
#Extract relevant information from each book item
def parse_book(item):
    volume = item.get("volumeInfo", {})
    
    return {
        "title": volume.get("title"),
        "authors": ", ".join(volume.get("authors", [])),
        "description": volume.get("description"),
        "categories": ", ".join(volume.get("categories", [])),
        "average_rating": volume.get("averageRating"),
        "ratings_count": volume.get("ratingsCount"),
        "published_year": volume.get("publishedDate", "")[:4]
    }

In [ ]:
#Collect data on multiple books from multiple genres
from tqdm import tqdm
import time

queries = [
    "fiction", "fantasy", "science fiction",
    "romance", "mystery", "history",
    "philosophy", "psychology"
]

all_books = []

for query in queries:
    for start in range(0, 200, 40):  # pagination
        params = {
            "q": query,
            "startIndex": start,
            "maxResults": 40,
            "key": api_key  # Include the API key from cell 1!
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        
        items = data.get("items", [])
        print(f"Query '{query}' at index {start}: Found {len(items)} books")
        
        for item in items:
            book = parse_book(item)
            all_books.append(book)
        
        time.sleep(1)  # avoid rate limits

print(f"Total books collected: {len(all_books)}")

Query 'fiction' at index 0: Found 20 books
Query 'fiction' at index 40: Found 20 books
Query 'fiction' at index 80: Found 20 books
Query 'fiction' at index 120: Found 20 books
Query 'fiction' at index 160: Found 20 books
Query 'fantasy' at index 0: Found 20 books
Query 'fantasy' at index 40: Found 20 books
Query 'fantasy' at index 80: Found 20 books
Query 'fantasy' at index 120: Found 20 books
Query 'fantasy' at index 160: Found 20 books
Query 'science fiction' at index 0: Found 20 books
Query 'science fiction' at index 40: Found 20 books
Query 'science fiction' at index 80: Found 20 books
Query 'science fiction' at index 120: Found 20 books
Query 'science fiction' at index 160: Found 20 books
Query 'romance' at index 0: Found 20 books
Query 'romance' at index 40: Found 20 books
Query 'romance' at index 80: Found 20 books
Query 'romance' at index 120: Found 20 books
Query 'romance' at index 160: Found 20 books
Query 'mystery' at index 0: Found 20 books
Query 'mystery' at index 40: Foun

In [11]:
#Convert data to dataframe  
import pandas as pd

df = pd.DataFrame(all_books)

# Clean up - remove duplicates first
df = df.drop_duplicates(subset=["title"])

# Only drop rows with missing descriptions if that column exists
if 'description' in df.columns:
    initial_rows = len(df)
    df = df.dropna(subset=["description"])
    removed = initial_rows - len(df)
    print(f"Removed {removed} rows without descriptions")

df.reset_index(drop=True, inplace=True)

print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Final shape: (0, 0)
Columns: []


""
